# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1 — "What Predicts Health?" (ML Appendix, p.27)**

Random Forest feature importance for predicting a composite Health Score, with
Average Position (43%) as the top feature, followed by Impressions (32%) and
Scroll Depth (15%).

My methodology question: where does the label (Health Score) come from? The paper
itself already discloses that Health Score is partly constructed from some of the
same inputs used as features (position and impressions are two of its four
components) — this is the exact label-derived-feature pattern I found in my own
ML-08/ML-09 work with clicks_first_half and ctr_first_half. I'd ask: could the paper
quantify how much of that 43% importance is mechanically guaranteed by the scoring
formula itself, versus genuine signal — e.g., by testing feature importance on a
health-score variant that excludes position/impressions from its own construction?
This is a constructive extension, not a correction — the paper already names the
issue more honestly than most public reports would.

**Finding 2 — "What Predicts Growth?" (ML Appendix, p.29)**

Logistic regression, reported at 71% holdout accuracy, separating growing from
declining pages using features like content age, days since update, and days visible.

My methodology question: does the validation design carry the 71% claim? The paper
doesn't specify whether the holdout split was random or grouped by brand. With 57
brands in the dataset, if pages from the same brand can appear in both the training
set and the holdout set, the model could partly be learning brand-level patterns
rather than genuine per-page growth signals — exactly the gap I tested for in my own
Section 2 (random vs. client-grouped split). I'd ask whether the holdout was grouped
by brand, and if not, whether re-running it that way changes the 71% figure.

In [6]:
print("Finding 1: 'What Predicts Health?' (p.27) — Random Forest, Health Score prediction")
print("Methodology question: does the label's partial construction from feature inputs")
print("(position, impressions) inflate importance mechanically? Paper already discloses this.\n")

print("Finding 2: 'What Predicts Growth?' (p.29) — Logistic Regression, 71% holdout accuracy")
print("Methodology question: was the holdout split grouped by brand (57 brands total),")
print("or random? A random split risks brand-level memorization inflating the accuracy —")
print("the same gap I tested for my own model in Section 2.")

Finding 1: 'What Predicts Health?' (p.27) — Random Forest, Health Score prediction
Methodology question: does the label's partial construction from feature inputs
(position, impressions) inflate importance mechanically? Paper already discloses this.

Finding 2: 'What Predicts Growth?' (p.29) — Logistic Regression, 71% holdout accuracy
Methodology question: was the holdout split grouped by brand (57 brands total),
or random? A random split risks brand-level memorization inflating the accuracy —
the same gap I tested for my own model in Section 2.


## 2. My model under an honest split (before/after)

Re-ran my Week-5 Random Forest under two splits: a naive random split (BEFORE) and
a grouped-by-client split (AFTER, same as ML-08's client_holdout approach).

The random split had 41 clients appearing in both train and test — a real leakage
risk, since the model could learn client-specific patterns and get an unfair
advantage on "unseen" rows from clients it already saw. The grouped split correctly
reduced this overlap to 0.

Surprisingly, Precision@10 stayed at 0.9 in both cases — the split type didn't
explain the earlier suspiciously-high scores. This points to the feature-label
shortcut found in ML-08 (clicks_first_half==0 mechanically forcing the label to 0)
as the dominant driver, not split contamination. Investigated formally in Section 3.

In [7]:
!pip install -q huggingface_hub datasets pandas pyarrow scikit-learn

from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MONTH = "2026-03"
ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files=f"fact_content_daily_performance/month={MONTH}/*.parquet",
    split="train", token=HF_TOKEN,
)
df_month = ds.to_pandas()
df_month["report_date"] = pd.to_datetime(df_month["report_date"])
df_month = df_month[df_month["gsc_data_available"] == True].copy()

first_half = df_month[df_month["report_date"] <= "2026-03-15"]
second_half = df_month[df_month["report_date"] >= "2026-03-16"]
group_cols = ["client_hash_id", "content_hash_id"]

first_agg = first_half.groupby(group_cols).agg(
    impressions_first_half=("gsc_impressions", "sum"),
    clicks_first_half=("gsc_clicks", "sum"),
    avg_position_first_half=("gsc_avg_position", "mean"),
).reset_index()
second_agg = second_half.groupby(group_cols).agg(
    clicks_second_half=("gsc_clicks", "sum"),
).reset_index()

pair_df = first_agg.merge(second_agg, on=group_cols, how="inner")
pair_df["ctr_first_half"] = (pair_df["clicks_first_half"] / pair_df["impressions_first_half"].replace(0, np.nan)) * 100
pair_df["is_declining_proxy"] = (pair_df["clicks_second_half"] < pair_df["clicks_first_half"]).astype(int)

features = ["impressions_first_half", "clicks_first_half", "ctr_first_half", "avg_position_first_half"]

def precision_at_k(df, score_col, k, label_col="is_declining_proxy"):
    return df.sort_values(score_col, ascending=False).head(k)[label_col].mean()

# --- BEFORE: naive random split (rows shuffled, ignoring client) ---
train_r, test_r = train_test_split(pair_df, test_size=0.3, random_state=42)
X_tr, y_tr = train_r[features].fillna(0), train_r["is_declining_proxy"]
X_te = test_r[features].fillna(0)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_tr, y_tr)
test_r = test_r.copy()
test_r["score"] = rf_random.predict_proba(X_te)[:, 1]
p10_random = precision_at_k(test_r, "score", 10)
base_random = test_r["is_declining_proxy"].mean()

# --- AFTER: grouped split (same as ML-08, client_holdout) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(pair_df, groups=pair_df["client_hash_id"]))
train_g, test_g = pair_df.iloc[train_idx].copy(), pair_df.iloc[test_idx].copy()
X_tr_g, y_tr_g = train_g[features].fillna(0), train_g["is_declining_proxy"]
X_te_g = test_g[features].fillna(0)
rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_tr_g, y_tr_g)
test_g["score"] = rf_grouped.predict_proba(X_te_g)[:, 1]
p10_grouped = precision_at_k(test_g, "score", 10)
base_grouped = test_g["is_declining_proxy"].mean()

overlap_random = set(train_r["client_hash_id"]) & set(test_r["client_hash_id"])
overlap_grouped = set(train_g["client_hash_id"]) & set(test_g["client_hash_id"])

comparison = pd.DataFrame({
    "Split type": ["Random (BEFORE)", "Grouped by client (AFTER)"],
    "Client overlap train/test": [len(overlap_random), len(overlap_grouped)],
    "Base rate": [base_random, base_grouped],
    "Precision@10": [p10_random, p10_grouped],
})
print(comparison.to_string(index=False))

               Split type  Client overlap train/test  Base rate  Precision@10
          Random (BEFORE)                         41    0.20221           0.9
Grouped by client (AFTER)                          0    0.24777           0.9


## 3. Leakage audit

Formal leakage audit using the attack checklist. Confirmed timeline is clean (features
strictly before the label window), no product flags used, grouped split verified with
zero client overlap, base rate reported next to every metric.

Ran the train-with/train-without test on the suspect feature. My first attempt only
removed clicks_first_half and saw NO collapse (0.900 -> 0.900) — which would have been
a false "all clear." Investigating further, I realized ctr_first_half is a SIBLING
feature (ctr = clicks / impressions), derived from the same raw count that mechanically
forces the label to 0 when clicks_first_half == 0. Removing BOTH sibling columns
produced a real collapse: 0.900 -> 0.300 — a genuine confession of leakage.

Lesson: checking only the obviously-named suspect column isn't enough — any feature
mathematically derived from a leaky source carries the same leak, even with a
different name.

In [8]:
from sklearn.metrics import roc_auc_score

pair_df["position_tier"] = pd.cut(pair_df["avg_position_first_half"], bins=[0,3,10,20,100], labels=["1-3","4-10","11-20","21+"])

# --- Attack checklist ---
print("ATTACK CHECKLIST\n")

# 1. Timeline: features strictly before label window?
print("[1] Timeline: features from Mar 1-15, label from Mar 16-31 (clicks_second_half < clicks_first_half).")
print("    Features are strictly before the label window. PASS.\n")

# 2. Label-derived features test: train WITH vs WITHOUT the suspect column
suspect = "clicks_first_half"
X_with = train_g[features].fillna(0)
X_without = train_g[[f for f in features if f != suspect]].fillna(0)
y = train_g["is_declining_proxy"]

rf_with = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_with, y)
rf_without = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_without, y)

score_with = precision_at_k(test_g.assign(score=rf_with.predict_proba(test_g[features].fillna(0))[:,1]), "score", 10)
score_without = precision_at_k(test_g.assign(score=rf_without.predict_proba(test_g[[f for f in features if f != suspect]].fillna(0))[:,1]), "score", 10)

print(f"[2] Train WITH '{suspect}': Precision@10 = {score_with:.3f}")
print(f"    Train WITHOUT '{suspect}': Precision@10 = {score_without:.3f}")
print(f"    Collapse of {score_with - score_without:.3f} confirms this feature drives the")
print(f"    suspiciously-high score — matches ML-08's finding about the clicks_first_half==0 shortcut.\n")

# 3. Product flags as features?
print("[3] Product flags / existing-system scores as features: NONE used. PASS.\n")

# 4. Grouped split?
print(f"[4] Split grouped by client_hash_id: YES (0 client overlap, confirmed in Section 2). PASS.\n")

# 5. Base rate printed next to metric?
print(f"[5] Base rate (grouped test set): {base_grouped:.3f}, vs Precision@10 = {p10_grouped:.3f}. PASS — reported together.\n")

# 6. Top feature sanity check
importances = pd.Series(rf_grouped.feature_importances_, index=features).sort_values(ascending=False)
print(f"[6] Top feature: {importances.index[0]} ({importances.iloc[0]:.3f}). This is suspiciously")
print(f"    high given the label-formula shortcut — investigated, not celebrated. FLAGGED.\n")

# 7. Metrics out-of-fold (test set never touched during training)?
print("[7] All Precision@10/base-rate numbers computed on held-out test_g only, never train_g. PASS.")

ATTACK CHECKLIST

[1] Timeline: features from Mar 1-15, label from Mar 16-31 (clicks_second_half < clicks_first_half).
    Features are strictly before the label window. PASS.

[2] Train WITH 'clicks_first_half': Precision@10 = 0.900
    Train WITHOUT 'clicks_first_half': Precision@10 = 0.900
    Collapse of 0.000 confirms this feature drives the
    suspiciously-high score — matches ML-08's finding about the clicks_first_half==0 shortcut.

[3] Product flags / existing-system scores as features: NONE used. PASS.

[4] Split grouped by client_hash_id: YES (0 client overlap, confirmed in Section 2). PASS.

[5] Base rate (grouped test set): 0.248, vs Precision@10 = 0.900. PASS — reported together.

[6] Top feature: ctr_first_half (0.492). This is suspiciously
    high given the label-formula shortcut — investigated, not celebrated. FLAGGED.

[7] All Precision@10/base-rate numbers computed on held-out test_g only, never train_g. PASS.


In [9]:
# --- Corrected test: ctr_first_half is a SIBLING of clicks_first_half (ctr = clicks/impressions) ---
# Removing clicks_first_half alone didn't collapse the score — ctr_first_half carried
# the same shortcut. Testing removal of BOTH.
suspects = ["clicks_first_half", "ctr_first_half"]
X_without_both = train_g[[f for f in features if f not in suspects]].fillna(0)
rf_without_both = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_without_both, y)
score_without_both = precision_at_k(
    test_g.assign(score=rf_without_both.predict_proba(test_g[[f for f in features if f not in suspects]].fillna(0))[:,1]),
    "score", 10
)

print(f"Train WITH both clicks_first_half + ctr_first_half: Precision@10 = {score_with:.3f}")
print(f"Train WITHOUT both (only impressions_first_half, avg_position_first_half): Precision@10 = {score_without_both:.3f}")
print(f"\nCollapse of {score_with - score_without_both:.3f} — THIS confirms the real leak:")
print("ctr_first_half is a sibling of clicks_first_half (both derived from the same raw count")
print("that mechanically determines the label when it's zero). My first test only removed one")
print("of the two sibling columns, which is why it looked clean — a good lesson in checking")
print("ALL columns derived from a suspect, not just the one with the obvious name.")

Train WITH both clicks_first_half + ctr_first_half: Precision@10 = 0.900
Train WITHOUT both (only impressions_first_half, avg_position_first_half): Precision@10 = 0.300

Collapse of 0.600 — THIS confirms the real leak:
ctr_first_half is a sibling of clicks_first_half (both derived from the same raw count
that mechanically determines the label when it's zero). My first test only removed one
of the two sibling columns, which is why it looked clean — a good lesson in checking
ALL columns derived from a suspect, not just the one with the obvious name.


## 4. Claim rewrite

My boldest unaudited claim would have been "Precision@10 = 0.900, ~3x the base rate"
— exactly the kind of headline number Section 3's audit exists to catch before it
gets published. The honest, leakage-free number is Precision@10 = 0.300 vs base rate
0.248: observed, modest, decision-support only — real signal, not a dramatic result.

In [10]:
bold_claim = "My Random Forest model achieves Precision@10 = 0.900, roughly 3x the base rate of 0.248."

safe_claim = (
    "On this March 2026 slice, the model's observed Precision@10 was 0.900 under a "
    "client-grouped split — but a leakage audit found this was driven almost entirely "
    "by ctr_first_half and clicks_first_half acting as sibling proxies for the label's "
    "own zero-click boundary condition, not genuine decline prediction. With both "
    "removed, the honest, decision-support-only Precision@10 is 0.300 against a base "
    "rate of 0.248 — a real but modest, directional signal, not a 3x improvement."
)

print("BEFORE (bold, unaudited):")
print(bold_claim)
print("\nAFTER (safe, audited):")
print(safe_claim)

BEFORE (bold, unaudited):
My Random Forest model achieves Precision@10 = 0.900, roughly 3x the base rate of 0.248.

AFTER (safe, audited):
On this March 2026 slice, the model's observed Precision@10 was 0.900 under a client-grouped split — but a leakage audit found this was driven almost entirely by ctr_first_half and clicks_first_half acting as sibling proxies for the label's own zero-click boundary condition, not genuine decline prediction. With both removed, the honest, decision-support-only Precision@10 is 0.300 against a base rate of 0.248 — a real but modest, directional signal, not a 3x improvement.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.